In [ ]:
import pandas as pd
file_path='/content/amazon.csv'
try:
  df=pd.read_csv(file_path)
  print("Dataset loaded Successfully!")
except FileNotFoundError:
  print(f"Error: The file '{file_path}' was not found")
except Exception as e:
  print("An unexpected error occurred.")

Dataset loaded Successfully!


This initial loading and inspection phase is fundamental. It gives us a clear picture of what we're working with and highlights the immediate areas that require attention for data cleaning.

In [ ]:
print("First 5 rows:")
print(df.head())
print("Dataframe info")
print(df.info())
print(f"Dataframe shape:{df.shape}")
print("Column names")
print(df.columns)


First 5 rows:
   product_id                                       product_name  \
0  B07JW9H4J1  Wayona Nylon Braided USB to Lightning Fast Cha...   
1  B098NS6PVG  Ambrane Unbreakable 60W / 3A Fast Charging 1.5...   
2  B096MSW6CT  Sounce Fast Phone Charging Cable & Data Sync U...   
3  B08HDJ86NZ  boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...   
4  B08CF3B7N1  Portronics Konnect L 1.2M Fast Charging 3A 8 P...   

                                            category discounted_price  \
0  Computers&Accessories|Accessories&Peripherals|...             ₹399   
1  Computers&Accessories|Accessories&Peripherals|...             ₹199   
2  Computers&Accessories|Accessories&Peripherals|...             ₹199   
3  Computers&Accessories|Accessories&Peripherals|...             ₹329   
4  Computers&Accessories|Accessories&Peripherals|...             ₹154   

  actual_price discount_percentage rating rating_count  \
0       ₹1,099                 64%    4.2       24,269   
1         ₹349        

Handling Missing values

In [ ]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 0
rating_count           2
about_product          0
user_id                0
user_name              0
review_id              0
review_title           0
review_content         0
img_link               0
product_link           0
dtype: int64


Drop rows where 'review' is missing

In [ ]:
initial_rows=df.shape[0]
df.dropna(subset=['review_title','review_content'],how='all',inplace=True)
rows_afterdropping_na=df.shape[0]
print(f"Dropped {initial_rows - rows_afterdropping_na} rows with missing review.")
print(f"DataFrame shape after dropping NA: {df.shape}")
#verify no more missing values present
print("Missing values after dropping NA in review:")
print(df.isnull().sum())

Dropped 0 rows with missing review.
DataFrame shape after dropping NA: (1465, 16)
Missing values after dropping NA in review:
product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 0
rating_count           2
about_product          0
user_id                0
user_name              0
review_id              0
review_title           0
review_content         0
img_link               0
product_link           0
dtype: int64


Remove Duplicate Reviews

In [ ]:
duplicated_rows_count=df.duplicated().sum()
print(f"Number of duplicated rows: {duplicated_rows_count}")
df.drop_duplicates(inplace=True)
print(f"DataFrame shape after dropping duplicates: {df.shape}")

Number of duplicated rows: 0
DataFrame shape after dropping duplicates: (1465, 16)


Text Cleaning

In [ ]:
# Remove leading/trailing whitespace
df['review_title'] = df['review_title'].str.strip()
df['review_content'] = df['review_content'].str.strip()

# Convert to lowercase
df['review_title'] = df['review_title'].str.lower()
df['review_content'] = df['review_content'].str.lower()

print("\nSample cleaned reviews (first 5):")
for i in range(5):
    print(f"{i+1} Title: {df['review_title'].iloc[i]}")
    print(f"   Content: {df['review_content'].iloc[i][:100]}...\n")


Sample cleaned reviews (first 5):
1 Title: satisfied,charging is really fast,value for money,product review,good quality,good product,good product,as of now seems good
   Content: looks durable charging is fine toono complains,charging is really fast, good product.,till now satis...

2 Title: a good braided cable for your type c device,good quality product from ambrane,super cable,as,good quality,good product,its good,good quality for the price but one issue with my unit
   Content: i ordered this cable to connect my phone to android auto of car. the cable is really strong and the ...

3 Title: good speed for earlier versions,good product,working good,good for the price,good,worth for money,working nice,it's a really nice product
   Content: not quite durable and sturdy,https://m.media-amazon.com/images/w/webp_402378-t1/images/i/71riggrbucl...

4 Title: good product,good one,nice,really nice product,very first time change,good,fine product but could be better,very nice it's charging l

Advanced Text Cleaning(Punctuation, Numbers, Special Characters)

In [ ]:
import re

# Combine title and content into one column
df['review_text'] = df['review_title'] + " " + df['review_content']

def remove_html_tags(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_special_characters(text):
    return re.sub(r'[^a-zA-Z\s]', '', text)

print("Applying advanced text cleaning...")

# Apply cleaning steps
df['review_text'] = df['review_text'].apply(remove_html_tags)
df['review_text'] = df['review_text'].apply(remove_punctuation)
df['review_text'] = df['review_text'].apply(remove_numbers)
df['review_text'] = df['review_text'].apply(remove_special_characters)

print("Advanced text cleaning complete.")

# Show sample
print("\nSample of cleaned review text (first 5):")
for i in range(5):
    print(f"{i+1}: {df['review_text'].iloc[i][:100]}...")

Applying advanced text cleaning...
Advanced text cleaning complete.

Sample of cleaned review text (first 5):
1: satisfiedcharging is really fastvalue for moneyproduct reviewgood qualitygood productgood productas ...
2: a good braided cable for your type c devicegood quality product from ambranesuper cableasgood qualit...
3: good speed for earlier versionsgood productworking goodgood for the pricegoodworth for moneyworking ...
4: good productgood onenicereally nice productvery first time changegoodfine product but could be bette...
5: as good as originaldecentgood one for secondary usebest qualitygoodamazing product at a mind blowing...


Text Processing - Tokenization,StopWord Removal, Stemming/Lemmalization

In [ ]:
import nltk
nltk.download('punkt') # For tokenization
nltk.download('punkt_tab')
nltk.download('stopwords') # For stop word removal
nltk.download('wordnet') # For lemmatization
nltk.download('omw-1.4') # For lemmatization
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Define the Comprehensive Preprocessing Function

In [ ]:
def preprocess_text(text):
    "Applies tokenization, stop word removal, and lemmatization to a given text.Assumes the input text has already undergone basic cleaning (lowercase, no punctuation/numbers)."
    # 1. Tokenization
    tokens = word_tokenize(text)

    # 2. Stop Word Removal and Lemmatization
    processed_tokens = []
    for word in tokens:
        if word not in stop_words:
            # Lemmatize the word
            lemma = lemmatizer.lemmatize(word)
            processed_tokens.append(lemma)

    # Join the processed tokens back into a string
    return ' '.join(processed_tokens)

print("Preprocessing function 'preprocess_text' defined.")

Preprocessing function 'preprocess_text' defined.


Apply Preprocessing to the DataFrame

In [ ]:
if 'df' in locals() and df is not None:
    print(f"--- Applying Text Preprocessing ---")
    print(f"Applying preprocessing to {len(df)} reviews...")

    df['processed_review_text'] = df['review_text'].apply(preprocess_text)

    print("Text preprocessing complete.")

    # Display samples of the original cleaned text and the newly processed text
    print("--- Sample Comparison ---")
    print("Original Cleaned Text (first 5):")
    for i in range(min(5, len(df))):
        print(f"{i+1}: {df['review_text'].iloc[i][:100]}...")

    print("Preprocessed Text (first 5):")
    for i in range(min(5, len(df))):
        print(f"{i+1}: {df['processed_review_text'].iloc[i][:100]}...")

    print(f"DataFrame shape after adding processed text: {df.shape}")
    print("DataFrame columns after preprocessing:")
    print(df.columns)

else:
    print("DataFrame 'df' not found or is None. Please ensure previous cleaning steps were executed successfully.")

--- Applying Text Preprocessing ---
Applying preprocessing to 1465 reviews...
Text preprocessing complete.
--- Sample Comparison ---
Original Cleaned Text (first 5):
1: satisfiedcharging is really fastvalue for moneyproduct reviewgood qualitygood productgood productas ...
2: a good braided cable for your type c devicegood quality product from ambranesuper cableasgood qualit...
3: good speed for earlier versionsgood productworking goodgood for the pricegoodworth for moneyworking ...
4: good productgood onenicereally nice productvery first time changegoodfine product but could be bette...
5: as good as originaldecentgood one for secondary usebest qualitygoodamazing product at a mind blowing...
Preprocessed Text (first 5):
1: satisfiedcharging really fastvalue moneyproduct reviewgood qualitygood productgood productas seems g...
2: good braided cable type c devicegood quality product ambranesuper cableasgood qualitygood productits...
3: good speed earlier versionsgood productworking goodgo

Prepare Data for Machine Learning

In [ ]:
print("--- Preparing Data for Machine Learning ---")

# Convert rating to numeric
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# Input features
X = df['processed_review_text']

# Target variable
y = df['rating']

print("Input and target variables prepared.")

--- Preparing Data for Machine Learning ---
Input and target variables prepared.


In [ ]:
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

df = df.dropna(subset=['rating'])

X = df['processed_review_text']
y = df['rating']

In [ ]:
print("--- Converting Ratings to Sentiment Labels ---")

df['sentiment'] = df['rating'].apply(
    lambda x: "positive" if x >= 4 else ("neutral" if x == 3 else "negative")
)

print(df['sentiment'].value_counts())

--- Converting Ratings to Sentiment Labels ---
sentiment
positive    1110
negative     350
neutral        4
Name: count, dtype: int64


In [ ]:
y=df['sentiment']
x=df['about_product']

Split Dataset into Training and Testing Sets

In [ ]:
from sklearn.model_selection import train_test_split

print("--- Splitting Dataset ---")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

--- Splitting Dataset ---
Training samples: 1024
Testing samples: 440


Apply TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("--- Applying TF-IDF Vectorization ---")

vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english'
)

# Fit on training data
X_train_tfidf = vectorizer.fit_transform(X_train)

# Transform test data
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF vectorization complete.")

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

--- Applying TF-IDF Vectorization ---
TF-IDF vectorization complete.
Training TF-IDF shape: (1024, 1000)
Testing TF-IDF shape: (440, 1000)


Train the Naive Bayes Model

In [ ]:
from sklearn.naive_bayes import MultinomialNB

print("--- Training Naive Bayes Model ---")

model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

print("Model training completed.")

--- Training Naive Bayes Model ---
Model training completed.


Make Predictions

In [ ]:
print("--- Making Predictions ---")

predictions = model.predict(X_test_tfidf)

print("Predictions completed.")

print("First 10 predictions:")
print(predictions[:10])

--- Making Predictions ---
Predictions completed.
First 10 predictions:
['negative' 'positive' 'positive' 'positive' 'positive' 'positive'
 'positive' 'positive' 'positive' 'positive']


Predict Sentiment for all reviews

In [ ]:
print("--- Predicting Sentiment for All Reviews ---")

all_reviews_tfidf = vectorizer.transform(df['processed_review_text'])

df['predicted_sentiment'] = model.predict(all_reviews_tfidf)

print("Predicted sentiment added to dataframe.")

--- Predicting Sentiment for All Reviews ---
Predicted sentiment added to dataframe.


Get Prediction Confidence Scores

In [ ]:
print("--- Calculating Sentiment Scores ---")

probabilities = model.predict_proba(all_reviews_tfidf)

sentiment_scores = probabilities.max(axis=1)

df['sentiment_score'] = sentiment_scores

print("Sentiment scores added to dataframe.")

--- Calculating Sentiment Scores ---
Sentiment scores added to dataframe.


Import SQLite Module and Define Database Name

In [ ]:
import sqlite3
DATABASE_NAME = 'ecommerce_sentiment.db'#configuration
print(f"SQLite module imported. Database file will be named: {DATABASE_NAME}")

SQLite module imported. Database file will be named: ecommerce_sentiment.db


 Create SQLite Database and Connection

In [ ]:
# Connect to the SQLite database. If the file does not exist, it will be created.
try:
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    print(f"Successfully connected to SQLite database: {DATABASE_NAME}")

    # Optional: Check if tables already exist and drop them if necessary for a clean start
    # In a real application, you'd handle this more robustly (e.g., migrations)
    cursor.execute("DROP TABLE IF EXISTS reviews;")
    cursor.execute("DROP TABLE IF EXISTS sentiment_predictions;")
    print("Dropped existing 'reviews' and 'sentiment_predictions' tables if they existed.")

except sqlite3.Error as e:
    print(f"Error connecting to SQLite database: {e}")
    conn = None # Ensure conn is None if connection fails
    cursor = None

# Proceed only if connection was successful
if conn:
    print("Database connection established.")
else:
    print("Database connection failed. Cannot proceed with table creation.")

Successfully connected to SQLite database: ecommerce_sentiment.db
Dropped existing 'reviews' and 'sentiment_predictions' tables if they existed.
Database connection established.


Define and Create the 'reviews' Table

In [ ]:
if cursor:
    print("--- Creating 'reviews' Table ---")

    # SQL statement to create the 'reviews' table
    # We include columns for original data and processed text.
    create_reviews_table_sql = """
    CREATE TABLE reviews (
        review_id INTEGER PRIMARY KEY,
        product_id TEXT,
        original_review_text TEXT,
        cleaned_review_text TEXT,
        processed_review_text TEXT,
        rating INTEGER,
        timestamp TEXT
    );
    """
    try:
        cursor.execute(create_reviews_table_sql)
        conn.commit()
        print("Table 'reviews' created successfully.")
    except sqlite3.Error as e:
        print(f"Error creating 'reviews' table: {e}")
else:
    print("Skipping 'reviews' table creation due to failed database connection.")

--- Creating 'reviews' Table ---
Table 'reviews' created successfully.


Define and Create the 'sentiment_predictions' Table

In [ ]:
if cursor:
    print("--- Creating 'sentiment_predictions' Table ---")

    # SQL statement to create the 'sentiment_predictions' table
    # This table will link predictions back to the original reviews.
    create_predictions_table_sql = """
    CREATE TABLE sentiment_predictions (
        prediction_id INTEGER PRIMARY KEY AUTOINCREMENT,
        review_id INTEGER,
        predicted_sentiment_label TEXT,
        predicted_sentiment_score REAL,
        FOREIGN KEY (review_id) REFERENCES reviews (review_id)
    );
    """
    try:
        cursor.execute(create_predictions_table_sql)
        conn.commit()
        print("Table 'sentiment_predictions' created successfully.")
    except sqlite3.Error as e:
        print(f"Error creating 'sentiment_predictions' table: {e}")
else:
    print("Skipping 'sentiment_predictions' table creation due to failed database connection.")

--- Creating 'sentiment_predictions' Table ---
Table 'sentiment_predictions' created successfully.


Close the Database Connection

In [ ]:
if conn:
    conn.close()
    print(f"Database connection closed.")
else:
    print("No active database connection to close.")

Database connection closed.


In [ ]:
import joblib

joblib.dump(model, "sentiment_model.joblib")
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")

print("Model and vectorizer saved successfully.")